In [1]:
import sys
sys.path.insert(0, r"E:\Claude Projects\quant_ict_trader")

import importlib
import backtests.backtest as bt_module
importlib.reload(bt_module)
from backtests.backtest import Backtest

bt = Backtest(
    instrument="EURUSD",
    start="2025-01-01",
    end="2025-06-30",
    account_balance=10000,
    risk_pct=0.01,
    tp1_rr=1.0,
    tp2_rr=2.0,
    sl_buffer_pips=3.0,
    min_rr=1.5,
    step_bars=24,
    min_warmup_bars=200,
)

result = bt.run()
bt.report()

KeyboardInterrupt: 

In [9]:
import sys
sys.path.insert(0, r"E:\Claude Projects\quant_ict_trader")
import yfinance as yf
import pandas as pd
from strategies.entry_model import EntryModel
from strategies.market_structure import MarketStructure

# Download data
raw_1h = yf.download("EURUSD=X", period="730d", interval="1h", auto_adjust=True, progress=False)
raw_4h = yf.download("EURUSD=X", period="730d", interval="4h", auto_adjust=True, progress=False)

def clean(df):
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0).str.lower()
    else:
        df.columns = df.columns.str.lower()
    df.index = pd.to_datetime(df.index, utc=True)
    return df.dropna()

df_1h = clean(raw_1h)
df_4h = clean(raw_4h)

# Check trend alignment every 50 bars
start = pd.Timestamp("2025-01-01", tz="UTC")
df_period = df_1h[df_1h.index >= start]

print("Checking trend alignment...")
aligned = 0
for i in range(200, len(df_period), 50):
    bar = df_period.index[i]
    s1 = df_1h[df_1h.index <= bar]
    s4 = df_4h[df_4h.index <= bar]
    ms1 = MarketStructure(s1, 5)
    ms4 = MarketStructure(s4, 5)
    
    def trend(ms):
        if not ms.structure_events: return 0
        k = ms.structure_events[-1].kind
        return 1 if "bull" in k else -1
    
    t1 = trend(ms1)
    t4 = trend(ms4)
    match = "✅" if t1 == t4 and t1 != 0 else "❌"
    if t1 == t4 and t1 != 0:
        aligned += 1
    print(f"  {bar.date()} | 4H:{t4:+d} | 1H:{t1:+d} | {match}")

print(f"\nAligned bars: {aligned}")

Checking trend alignment...
  2025-01-14 | 4H:-1 | 1H:+1 | ❌
  2025-01-16 | 4H:-1 | 1H:+1 | ❌
  2025-01-20 | 4H:-1 | 1H:+1 | ❌
  2025-01-22 | 4H:+1 | 1H:+1 | ✅
  2025-01-24 | 4H:+1 | 1H:+1 | ✅
  2025-01-28 | 4H:+1 | 1H:-1 | ❌
  2025-01-30 | 4H:+1 | 1H:+1 | ✅
  2025-02-03 | 4H:-1 | 1H:-1 | ✅
  2025-02-05 | 4H:-1 | 1H:+1 | ❌
  2025-02-10 | 4H:-1 | 1H:-1 | ✅
  2025-02-12 | 4H:-1 | 1H:+1 | ❌
  2025-02-14 | 4H:+1 | 1H:+1 | ✅
  2025-02-18 | 4H:+1 | 1H:-1 | ❌
  2025-02-20 | 4H:+1 | 1H:+1 | ✅
  2025-02-24 | 4H:+1 | 1H:+1 | ✅
  2025-02-26 | 4H:+1 | 1H:-1 | ❌
  2025-02-28 | 4H:-1 | 1H:-1 | ✅
  2025-03-04 | 4H:+1 | 1H:+1 | ✅
  2025-03-06 | 4H:+1 | 1H:+1 | ✅
  2025-03-11 | 4H:+1 | 1H:-1 | ❌
  2025-03-13 | 4H:+1 | 1H:+1 | ✅
  2025-03-17 | 4H:+1 | 1H:-1 | ❌
  2025-03-19 | 4H:+1 | 1H:-1 | ❌
  2025-03-21 | 4H:+1 | 1H:-1 | ❌
  2025-03-25 | 4H:-1 | 1H:+1 | ❌
  2025-03-27 | 4H:-1 | 1H:+1 | ❌
  2025-03-31 | 4H:+1 | 1H:+1 | ✅
  2025-04-02 | 4H:+1 | 1H:+1 | ✅
  2025-04-07 | 4H:+1 | 1H:-1 | ❌
  2025-04-09 | 

In [11]:
import importlib
import backtests.backtest as bt_module
importlib.reload(bt_module)
from backtests.backtest import Backtest

bt = Backtest(
    instrument="EURUSD",
    start="2025-01-01",
    end="2025-06-30",
    account_balance=10000,
    risk_pct=0.01,
    tp1_rr=1.0,
    tp2_rr=2.0,
    sl_buffer_pips=3.0,
    min_rr=1.5,
    step_bars=24,
    min_warmup_bars=200,
)

result = bt.run()
bt.report()


Backtest: EURUSD  2025-01-01 → 2025-06-30
1H bars: 17254  4H bars: 4355
Bars in period: 3028
Running bar by bar simulation...
  Bar 200 | 2025-01-14 | htf:-1 | ltf:+1 | fvgs:14
  Bar 224 | 2025-01-15 | htf:-1 | ltf:+1 | fvgs:14
  Bar 248 | 2025-01-16 | htf:-1 | ltf:+1 | fvgs:13
  Bar 272 | 2025-01-17 | htf:-1 | ltf:+1 | fvgs:13
  ✅ Trade: long @ 1.02302 | SL:1.02639 | 2025-01-21
  ✅ Trade: long @ 1.02796 | SL:1.02639 | 2025-01-21
  ✅ Trade: long @ 1.02786 | SL:1.03425 | 2025-01-23
  ✅ Trade: long @ 1.03686 | SL:1.03425 | 2025-01-23
  ✅ Trade: long @ 1.04324 | SL:1.03855 | 2025-01-27
  ✅ Trade: short @ 1.11801 | SL:1.04698 | 2025-02-03
  ✅ Trade: short @ 1.11290 | SL:1.04698 | 2025-02-03
  ✅ Trade: short @ 1.09439 | SL:1.04436 | 2025-02-07
  ✅ Trade: short @ 1.08868 | SL:1.04436 | 2025-02-07
  ✅ Trade: long @ 1.03130 | SL:1.02925 | 2025-02-13
  ✅ Trade: long @ 1.04058 | SL:1.02925 | 2025-02-13
  ✅ Trade: long @ 1.04357 | SL:1.03180 | 2025-02-17
  ✅ Trade: short @ 1.07788 | SL:1.05304 |

In [2]:
import sys
sys.path.insert(0, r"E:\Claude Projects\quant_ict_trader")

import yfinance as yf
test = yf.download("EURUSD=X", period="30d", interval="1h", auto_adjust=True, progress=False)
print(f"Data check: {len(test)} rows")

Data check: 697 rows


In [5]:
import sys
sys.path.insert(0, r"E:\Claude Projects\quant_ict_trader")

import yfinance as yf
import importlib

import backtests.backtest as bt_module
importlib.reload(bt_module)
from backtests.backtest import Backtest

import utils.signal_viewer as sv_module
importlib.reload(sv_module)
from utils.signal_viewer import plot_all_signals

from strategies.market_structure import MarketStructure

test = yf.download("EURUSD=X", period="30d", interval="1h", auto_adjust=True, progress=False)
print(f"Data check: {len(test)} rows")

bt = Backtest(
    instrument="EURUSD",
    start="2025-03-01",
    end="2025-04-30",
    account_balance=10000,
    risk_pct=0.01,
    tp1_rr=1.0,
    tp2_rr=2.0,
    sl_buffer_pips=3.0,
    min_rr=1.5,
    step_bars=24,
    min_warmup_bars=200,
)

result = bt.run()
bt.report()

for t in bt.trades:
    print(f"{t.direction} @ {t.entry_price:.5f} | TP1:{t.tp1:.5f} | TP2:{t.tp2:.5f} | RR:{t.signal.rr_tp2} | result:{t.result} | bars:{t.bars_held}")

bt_signals = [t.signal for t in bt.trades]
print(f"\nGenerating charts for {len(bt_signals)} trades...")

plot_all_signals(
    df=bt._df_1h,
    signals=bt_signals,
    candles_before=60,
    candles_after=40,
    save_html=r"E:\Claude Projects\quant_ict_trader\backtest_signals.html"
)

ms = MarketStructure(bt._df_1h, 5)
ms.plot(last_n=500, title="EURUSD 1H — Backtest Period").write_html(
    r"E:\Claude Projects\quant_ict_trader\backtest_chart.html"
)

print("Done — open backtest_signals.html and backtest_chart.html in browser")

Data check: 697 rows

Backtest: EURUSD  2025-03-01 → 2025-04-30
1H bars: 17231  4H bars: 4356
Bars in period: 999
Running bar by bar simulation...
  Bar 200 | 2025-03-13 | htf:+1 | ltf:-1 | fvgs:18
  Bar 224 | 2025-03-14 | htf:+1 | ltf:-1 | fvgs:17
  Bar 248 | 2025-03-17 | htf:+1 | ltf:-1 | fvgs:17
  Bar 272 | 2025-03-18 | htf:+1 | ltf:+1 | fvgs:18
  ✅ Trade: long @ 1.09266 | SL:1.08926 | 2025-04-09
  ✅ Trade: long @ 1.09655 | SL:1.08926 | 2025-04-09
  ✅ Trade: long @ 1.12133 | SL:1.08926 | 2025-04-11
  ✅ Trade: long @ 1.13116 | SL:1.12494 | 2025-04-16
  ✅ Trade: long @ 1.14195 | SL:1.13413 | 2025-04-21
  ✅ Trade: long @ 1.13604 | SL:1.13079 | 2025-04-25

  BACKTEST RESULTS — EURUSD
  2025-03-01  →  2025-04-30
  Total trades   : 5
  Wins           : 3  (60.0%)
  Losses         : 1
  Partials (TP1) : 0
  Expired        : 1
───────────────────────────────────────────────────────
  Total P&L      : $+330.65
  Total profit   : $434.61
  Total loss     : $103.96
  Profit factor  : 4.18
────

In [5]:
import sys
sys.path.insert(0, r"E:\Claude Projects\quant_ict_trader")

import yfinance as yf
import importlib

import backtests.backtest as bt_module
importlib.reload(bt_module)
from backtests.backtest import Backtest

import utils.signal_viewer as sv_module
importlib.reload(sv_module)
from utils.signal_viewer import plot_all_signals

from strategies.market_structure import MarketStructure

test = yf.download("EURUSD=X", period="30d", interval="1h", auto_adjust=True, progress=False)
print(f"Data check: {len(test)} rows")

bt = Backtest(
    instrument="EURUSD",
    start="2026-04-19",
    end="2026-04-29",
    account_balance=10000,
    risk_pct=0.01,
    tp1_rr=1.3,
    tp2_rr=1.3,
    sl_buffer_pips=2.0,
    min_sl_pips=8.0,
    max_sl_pips=50.0,
    step_bars=1,
    min_warmup_bars=50,
    htf_interval="1h",
    ltf_interval="15m",
)

result = bt.run()
bt.report()

for t in bt.trades:
    print(f"{t.direction} @ {t.entry_price:.5f} | TP1:{t.tp1:.5f} | TP2:{t.tp2:.5f} | RR:{t.signal.rr_tp2} | result:{t.result} | bars:{t.bars_held}")

plot_all_signals(
    df=bt._df_1h,
    signals=[t.signal for t in bt.trades],
    trades=bt.trades,
    candles_before=60,
    candles_after=40,
    save_html=r"E:\Claude Projects\quant_ict_trader\backtest_signals.html"
)

MarketStructure(bt._df_1h, 5).plot(last_n=500, title="EURUSD 1H — Backtest").write_html(
    r"E:\Claude Projects\quant_ict_trader\backtest_chart.html"
)

print("Done")

Data check: 703 rows

Backtest: EURUSD  2026-04-19 → 2026-04-29
Timeframes: HTF=1h  LTF=15m
LTF bars: 5633  HTF bars: 17237
Bars in period: 658
Running bar by bar simulation...
  Bar 50 | 2026-04-20 | htf:-1 | ltf:+1 | fvgs:18
  Bar 51 | 2026-04-20 | htf:-1 | ltf:+1 | fvgs:18
  Bar 52 | 2026-04-20 | htf:-1 | ltf:+1 | fvgs:18
  Bar 53 | 2026-04-20 | htf:-1 | ltf:+1 | fvgs:18


KeyboardInterrupt: 

In [7]:
import sys
sys.path.insert(0, r"E:\Claude Projects\quant_ict_trader")

import pandas as pd
import yfinance as yf
from strategies.market_structure import MarketStructure
from strategies.fvg import FairValueGap

# Download 15min data
raw = yf.download("EURUSD=X", period="60d", interval="15m", auto_adjust=True, progress=False)
df = raw.copy()
if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.get_level_values(0).str.lower()
df.index = pd.to_datetime(df.index, utc=True)
df = df.dropna()

# Check at a specific point
slice_df = df.iloc[:200]
ms = MarketStructure(slice_df, 5)
fvg = FairValueGap(slice_df, 1.0)

current_price = slice_df["close"].iloc[-1]
print(f"Current price: {current_price:.5f}")
print(f"Unfilled FVGs: {len(fvg.unfilled())}")

for f in fvg.unfilled():
    last_8 = slice_df.iloc[-8:]
    if f.kind == "bearish":
        touched = (last_8["high"] >= f.bottom).any() and (last_8["high"] <= f.top + 0.0003).any()
        print(f"  bearish FVG {f.bottom:.5f}-{f.top:.5f} | touched:{touched}")


for f in fvg.unfilled():
    last_8 = slice_df.iloc[-8:]
    dist = abs(current_price - f.midpoint) / 0.0001
    touched_high = (last_8["high"] >= f.bottom).any()
    touched_low = (last_8["low"] <= f.top).any()
    print(f"  {f.kind} | {f.bottom:.5f}-{f.top:.5f} | dist:{dist:.0f}p | high_touch:{touched_high} | low_touch:{touched_low}")

Current price: 1.18273
Unfilled FVGs: 5
  bullish | 1.17786-1.17813 | dist:47p | high_touch:True | low_touch:False
  bullish | 1.17952-1.17966 | dist:31p | high_touch:True | low_touch:False
  bullish | 1.17966-1.17980 | dist:30p | high_touch:True | low_touch:False
  bullish | 1.18008-1.18050 | dist:24p | high_touch:True | low_touch:False
  bullish | 1.18050-1.18064 | dist:22p | high_touch:True | low_touch:False
